# ISIC 2024 - 最終提交（LightGBM ＋ CNN 融合）

**Kaggle 設定**：
1. Input 加入比賽資料 `isic-2024-challenge`
2. Input 再加入 **`isic2024_cnn_train` notebook 的輸出**（Add Input → Your Work → 選那個 notebook）
3. Accelerator 選 **GPU**（測試集約 50 萬張要跑 CNN）
4. Internet **關掉**，Save Version，然後 Submit

流程：metadata 特徵 ＋ 病人內標準化 ＋ **CNN 預測當特徵** → LightGBM 5 折 → 測試集預測。

In [ ]:
import glob, io, os, time
import numpy as np
import pandas as pd
import h5py
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

KAGGLE_INPUT = '/kaggle/input'

def find_dir(fname):
    # Kaggle 的掛載路徑會變（例如多一層 competitions/），所以用搜尋的找檔案所在資料夾
    for depth in range(1, 4):
        hits = sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, fname)),
                      key=lambda p: ('isic-2024-challenge' not in p, p))  # 優先選比賽本身的資料
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'在 {KAGGLE_INPUT} 底下找不到 {fname}，請確認右側 Input 有加入')

DATA = find_dir('train-metadata.csv')
print('資料路徑:', DATA)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEVICE)

In [ ]:
# 自動找 CNN 訓練 notebook 的輸出（右側 Input 要先加入那個 notebook）
CNN_DIR = find_dir('oof_cnn.csv')
print('CNN 輸出路徑:', CNN_DIR)
CFG = dict(img=128, bs=128, workers=4, folds=5)
NEG_RATIO = 50
import lightgbm as lgb

In [ ]:
def pauc_above_tpr(y_true, y_pred, min_tpr=0.80):
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = 1.0 - np.asarray(y_pred)
    max_fpr = 1 - min_tpr
    scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    return 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (scaled - 0.5)

In [ ]:
class LesionDataset(Dataset):
    """從 hdf5 讀 JPEG bytes。每個 worker 各自開檔，避免 h5py 多程序問題。"""
    def __init__(self, h5_path, ids, targets=None, tfm=None):
        self.h5_path, self.ids, self.targets, self.tfm = h5_path, list(ids), targets, tfm
        self.h5 = None
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, 'r')
        img = Image.open(io.BytesIO(bytes(self.h5[self.ids[i]][()]))).convert('RGB')
        x = self.tfm(img)
        y = -1.0 if self.targets is None else float(self.targets[i])
        return x, torch.tensor(y, dtype=torch.float32)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tfm = T.Compose([
    T.Resize((CFG['img'], CFG['img'])),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(90),
    T.ColorJitter(0.2, 0.2, 0.1, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tfm = T.Compose([T.Resize((CFG['img'], CFG['img'])), T.ToTensor(), T.Normalize(MEAN, STD)])

def make_model(pretrained):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m

@torch.no_grad()
def predict(model, h5_path, ids):
    ds = LesionDataset(h5_path, ids, tfm=eval_tfm)
    dl = DataLoader(ds, batch_size=CFG['bs'] * 2, shuffle=False, num_workers=CFG['workers'])
    model.eval()
    out = []
    for x, _ in dl:
        out.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

## 讀資料、建特徵

In [ ]:
train = pd.read_csv(f'{DATA}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA}/test-metadata.csv')

drop = {'isic_id', 'patient_id', 'target', 'image_type', 'attribution', 'copyright_license'}
common = [c for c in test.columns if c in train.columns and c not in drop]
cat_cols = [c for c in common if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in common if c not in cat_cols]
for c in cat_cols:
    cats = pd.Categorical(train[c]).categories
    train[c] = pd.Categorical(train[c], categories=cats)
    test[c] = pd.Categorical(test[c], categories=cats)

pn_cols = [c for c in num_cols if c not in ('age_approx', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z')]
def add_patient_feats(df):
    g = df.groupby('patient_id')[pn_cols]
    z = (df[pn_cols] - g.transform('mean')) / (g.transform('std') + 1e-6)
    z.columns = [c + '_pnorm' for c in pn_cols]
    df = pd.concat([df, z], axis=1)
    df['patient_n_lesions'] = df.groupby('patient_id')['isic_id'].transform('count')
    return df
train, test = add_patient_feats(train), add_patient_feats(test)
base_features = num_cols + cat_cols + [c + '_pnorm' for c in pn_cols] + ['patient_n_lesions']

## CNN 預測當特徵（train 用 OOF，test 用 5 個 fold 模型平均）

In [ ]:
train = train.merge(pd.read_csv(f'{CNN_DIR}/oof_cnn.csv'), on='isic_id', how='left')
test_pred = np.zeros(len(test))
for fold in range(CFG['folds']):
    m = make_model(pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(f'{CNN_DIR}/cnn_fold{fold}.pt', map_location=DEVICE))
    test_pred += predict(m, f'{DATA}/test-image.hdf5', test.isic_id) / CFG['folds']
test['cnn_pred'] = test_pred

# 病人內的 CNN 預測排名：這顆是不是這個人最可疑的
for df in (train, test):
    df['cnn_pred_pnorm'] = df.groupby('patient_id')['cnn_pred'].rank(pct=True)
features = base_features + ['cnn_pred', 'cnn_pred_pnorm']
print(len(features), 'features')

## LightGBM 5 折（比較有沒有 CNN 特徵）

In [ ]:
params = dict(objective='binary', learning_rate=0.03, num_leaves=15, min_child_samples=40,
              colsample_bytree=0.5, subsample=0.8, subsample_freq=1, reg_lambda=5,
              n_estimators=200, random_state=SEED, verbose=-1)
sgkf = StratifiedGroupKFold(n_splits=CFG['folds'], shuffle=True, random_state=SEED)
folds = list(sgkf.split(train, train['target'], train['patient_id']))

def run_cv(feats):
    oof, models_ = np.zeros(len(train)), []
    for tr, va in folds:
        d = train.iloc[tr]
        pos = d[d.target == 1]
        neg = d[d.target == 0].sample(min(len(pos) * NEG_RATIO, int((d.target == 0).sum())), random_state=SEED)
        d = pd.concat([pos, neg])
        m = lgb.LGBMClassifier(**params).fit(d[feats], d.target)
        oof[va] = m.predict_proba(train.iloc[va][feats])[:, 1]
        models_.append(m)
    return pauc_above_tpr(train.target, oof), models_

score_base, _ = run_cv(base_features)
score_fused, models_fused = run_cv(features)
print(f'LightGBM（只有 metadata）  OOF pAUC = {score_base:.4f}')
print(f'LightGBM ＋ CNN 特徵        OOF pAUC = {score_fused:.4f}')

## 產生 submission.csv

In [ ]:
pred = np.mean([m.predict_proba(test[features])[:, 1] for m in models_fused], axis=0)
sub = pd.DataFrame({'isic_id': test['isic_id'], 'target': pred})
sub.to_csv('submission.csv', index=False)
sub.head()